# Анализ вакансий аналитиков данных

Блокнот работает с базой `vacancies.db`, которую каждый день пополняет
сборщик из репозитория [Fanraeen/vacancies-analysis](https://github.com/Fanraeen/vacancies-analysis).
Источник данных: портал «Работа в России» ([trudvsem.ru](https://trudvsem.ru)).

**Как запустить:** меню *Среда выполнения → Выполнить все*. Блокнот сам скачает свежую базу с GitHub.

Что внутри: загрузка данных → отбор аналитических вакансий → география → требования к опыту → навыки → зарплаты → динамика по дням.

## 1. Загрузка данных

In [ ]:
import re
import sqlite3
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/Fanraeen/vacancies-analysis/main/data/vacancies.db"

urllib.request.urlretrieve(DB_URL, "vacancies.db")
print("База скачана")

Если загрузка не сработала (например, репозиторий закрыт или изменился путь к файлу),
загрузите `vacancies.db` вручную: замените ячейку выше на

```python
from google.colab import files
files.upload()   # выберите data/vacancies.db с компьютера
```

In [ ]:
conn = sqlite3.connect("vacancies.db")
all_df = pd.read_sql("SELECT * FROM vacancies", conn)

snapshots = sorted(all_df["snapshot_date"].unique())
print(f"Строк в базе: {len(all_df)}")
print(f"Срезов: {len(snapshots)} (с {snapshots[0]} по {snapshots[-1]})")

last = snapshots[-1]
df_last = all_df[all_df["snapshot_date"] == last].copy()
print(f"Вакансий в последнем срезе ({last}): {len(df_last)}")

## 2. Отбор аналитических вакансий

Поиск по API идёт по тексту, поэтому в выдаче много лишнего (фармацевты, учётчики и т. д.).
Оставляем вакансии, в названии которых есть «аналитик» или «analyst».
Профессии на границе (экономисты, маркетологи) в анализ не входят.

In [ ]:
ANALYST = r"аналитик|analyst"


def only_analysts(d):
    return d[d["title"].fillna("").str.contains(ANALYST, case=False, regex=True)].copy()


def category(title):
    t = str(title).lower()
    if re.search(r"системн\w*\s+аналитик|system\w*\s+analyst", t):
        return "Системный аналитик"
    if re.search(r"бизнес\W*(?:/\w+\W*)?аналитик|business\s+analyst", t):
        return "Бизнес-аналитик"
    if re.search(r"аналитик\w*\s+(?:больших\s+)?данных|data\s+analyst|\bbi\b", t):
        return "Аналитик данных / BI"
    return "Другой аналитик"


df = only_analysts(df_last)
df["category"] = df["title"].apply(category)

print(f"Срез {last}: всего вакансий {len(df_last)}, аналитических {len(df)}\n")
print(df["category"].value_counts())

## 3. География

In [ ]:
top = df["region"].value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(top.index, top.values, color="#4C72B0")
ax.bar_label(bars, padding=3)
ax.set_title(f"Вакансии аналитиков по регионам (срез {last})")
ax.set_xlabel("Число вакансий")
fig.tight_layout()
plt.show()

share = df["region"].isin(["Город Москва", "Город Санкт-Петербург"]).mean()
print(f"Доля Москвы и Санкт-Петербурга: {share:.0%}")

> **Оговорка.** Регион в данных может быть регионом работодателя, а не места работы.
> Выводы относятся к этой базе, а не ко всему рынку.

## 4. Требования к опыту

Опыт извлекается из текста требований регулярным выражением (минимальное число лет
рядом со словом «опыт»). Метод неточный: например, «SQL от 1 года» относится к навыку,
а не к общему стажу, а текст в базе обрезан.

In [ ]:
def min_experience(text):
    """Минимальный опыт в годах из текста требований (None, если не нашли)."""
    if not isinstance(text, str) or not text.strip():
        return None
    t = text.lower()
    if re.search(r"без опыта|опыт[^.;]{0,20}не (?:требуется|обязателен|нужен)", t):
        return 0
    m = re.search(r"опыт[^.;]{0,80}?(\d+)(?:[\s\dх]|-|–)*(?:лет|год)", t)
    if m:
        return int(m.group(1))
    return None


def bucket(x):
    """Раскладывает число лет опыта по группам."""
    if isinstance(x, bool) or x is None or pd.isna(x):
        return "не указан"
    if x == 0:
        return "без опыта"
    if x <= 1:
        return "около 1 года"
    if x <= 3:
        return "2–3 года"
    return "4+ лет"


# быстрая проверка разбора на примерах
assert min_experience("Опыт работы более 3-х лет") == 3
assert min_experience("Опыт от 3-5 лет") == 3
assert min_experience("Без опыта") == 0
assert min_experience("Высшее образование") is None
assert min_experience(None) is None

df["exp_years"] = df["requirements"].apply(min_experience)
df["exp_bucket"] = df["exp_years"].apply(bucket)

order = ["без опыта", "около 1 года", "2–3 года", "4+ лет", "не указан"]
exp = df["exp_bucket"].value_counts().reindex(order).dropna()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(exp.index, exp.values, color="#55A868")
ax.bar_label(bars, padding=3)
ax.set_title(f"Требуемый опыт в вакансиях аналитиков (срез {last})")
ax.set_ylabel("Число вакансий")
fig.tight_layout()
plt.show()

has_text = df["requirements"].fillna("").str.strip().ne("").sum()
print(f"Текст требований есть у {has_text} из {len(df)} вакансий")

«Не указан» включает вакансии без текста требований и те, где опыт в сохранённом фрагменте
не назван. Группа «2–3 года» учитывает минимальное число лет (фраза «более 3 лет» тоже попадёт сюда).

### Вакансии с требованием опыта до 1 года

In [ ]:
low = df[df["exp_bucket"].isin(["без опыта", "около 1 года"])]
low = low.drop_duplicates(subset=["title", "company", "region"])

pd.set_option("display.max_colwidth", 250)
print(f"Найдено уникальных вакансий: {len(low)}")
cols = [c for c in ["title", "company", "region", "salary_min", "requirements", "vac_url"] if c in low.columns]
low[cols]

Перед откликом откройте полное описание вакансии: в нём бывают детали, которых нет в обрезанном
тексте. Пример ловушки: «SQL от 1 года» не означает, что общий опыт нужен всего год.

## 5. Навыки в требованиях

Считаем долю вакансий, где навык встречается в тексте требований (от числа вакансий **с текстом**).
Текст обрезан, поэтому реальные доли могут быть выше; разница в несколько процентов на такой выборке
может быть случайной.

In [ ]:
skills = {
    "Excel": r"excel|эксель",
    "SQL": r"sql",
    "Python": r"python",
    "BI-системы": r"power\s?bi|tableau|datalens|qlik|\bbi\b",
    "1С": r"1с|1c",
    "Статистика": r"статистик",
}

text = df["requirements"].fillna("").str.lower()
n_text = int((text.str.strip() != "").sum())

rows = []
for name, pat in skills.items():
    n = int(text.str.contains(pat, regex=True).sum())
    rows.append((name, n, n / n_text if n_text else 0))

skills_df = pd.DataFrame(rows, columns=["навык", "вакансий", "доля"]).sort_values("вакансий")
print(f"Вакансий с текстом требований: {n_text} из {len(df)}")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(skills_df["навык"], skills_df["вакансий"], color="#C44E52")
ax.bar_label(bars, padding=3)
ax.set_title("Навыки в требованиях (число вакансий с упоминанием)")
fig.tight_layout()
plt.show()

skills_df.sort_values("вакансий", ascending=False).assign(
    доля=lambda x: (x["доля"] * 100).round(0).astype(int).astype(str) + "%"
)

## 6. Зарплаты

Зарплата указана далеко не везде и, как правило, до вычета налогов. Смотрим нижнюю границу вилки.

In [ ]:
sal = df.loc[df["salary_min"] > 0, "salary_min"]
print(f"Зарплата указана у {len(sal)} из {len(df)} вакансий ({len(sal) / len(df):.0%})")

if len(sal):
    print(f"Медиана нижней границы: {sal.median():,.0f} руб.".replace(",", " "))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(sal / 1000, bins=12, color="#8172B3", edgecolor="white")
    ax.set_title("Нижняя граница зарплаты, тыс. руб.")
    ax.set_xlabel("тыс. руб.")
    ax.set_ylabel("Число вакансий")
    fig.tight_layout()
    plt.show()

## 7. Динамика по дням

Сборщик каждый день добавляет в базу новый срез, поэтому можно смотреть, как меняется число
аналитических вакансий. Пока срезов мало, график будет коротким.

In [ ]:
daily = only_analysts(all_df).groupby("snapshot_date")["id"].nunique()
print(daily)

if len(daily) > 1:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(daily.index, daily.values, marker="o")
    ax.set_title("Число аналитических вакансий по дням")
    ax.set_ylabel("Вакансий")
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()
    plt.show()
else:
    print("\nПока только один срез: график динамики появится, когда накопятся данные.")

## Ограничения данных

- Один источник (государственная база): крупные работодатели часто размещают вакансии на других площадках.
- Фильтр по названию, а не по содержанию вакансии.
- Регион может быть регионом работодателя.
- Текст требований обрезан, часть упоминаний опыта и навыков потеряна.
- Опыт и навыки извлекаются из текста регулярными выражениями и бывают неточными.
- Поле формата занятости заполнено не у всех вакансий, поэтому долю удалённой работы оценить нельзя.